In [1]:
import yaml
import os
import polars as pl
from datetime import datetime ,date

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

E:\smallcaps_data
['AIXI.parquet', 'APUS.parquet', 'BAOS.parquet', 'BEAT.parquet', 'BIAF.parquet', 'CAN.parquet', 'CDTG.parquet', 'CHAI.parquet', 'CHPT.parquet', 'CISS.parquet', 'COOT.parquet', 'CYAB.parquet', 'FBGL.parquet', 'FLYE.parquet', 'GCDT.parquet', 'GIPR.parquet', 'GRI.parquet', 'GRML.parquet', 'IMCC.parquet', 'IMRN.parquet', 'LIDR.parquet', 'logs', 'MEDS.parquet', 'NCPL.parquet', 'NCRA.parquet', 'NEOV.parquet', 'OFAL.parquet', 'OLB.parquet', 'OLOX.parquet', 'output', 'PDSB.parquet', 'PFSA.parquet', 'SBFM.parquet', 'SDEV.parquet', 'SKYQ.parquet', 'SOAR.parquet', 'SPHL.parquet', 'STAK.parquet', 'TNMG.parquet', 'TOPP.parquet', 'TRUG.parquet', 'USDE.parquet', 'VEEA.parquet', 'VRA.parquet', 'VRME.parquet', 'XHLD.parquet', 'XPON.parquet', 'XTNT.parquet', '_catalog.parquet']


In [2]:
# list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
list_of_tickers = [f for f in list_of_files if not f.startswith("_catalog") and f.endswith(".parquet")]
catalog = [f for f in list_of_files if f.startswith("_catalog") and f.endswith(".parquet")][0]
# parquet_path = os.path.join(data_dir, f"{ticker}.parquet")
catalog_path = os.path.join(data_dir, f"{catalog}")

df_catalog = pl.read_parquet(catalog_path)['ticker','end'].sort('ticker')
df_catalog = df_catalog.with_columns(
    pl.col('end').dt.date().alias('date')
)
df_catalog


ticker,end,date
str,"datetime[μs, UTC]",date
"""AIXI""",2026-08-07 23:59:00 UTC,2026-08-07
"""AIXI""",2026-08-28 23:59:00 UTC,2026-08-28
"""APUS""",2026-09-04 20:32:00 UTC,2026-09-04
"""APUS""",2026-09-24 23:59:00 UTC,2026-09-24
"""APUS""",2026-10-07 23:59:00 UTC,2026-10-07
…,…,…
"""XHLD""",2026-10-06 23:59:00 UTC,2026-10-06
"""XPON""",2026-08-04 20:05:00 UTC,2026-08-04
"""XPON""",2026-08-24 23:59:00 UTC,2026-08-24


In [3]:
df_backtest_runs = (
    df_catalog
    .select("ticker", "date")
    .unique()
    .sort(["ticker", "date"])
)
df_backtest_runs


ticker,date
str,date
"""AIXI""",2026-08-07
"""AIXI""",2026-08-28
"""APUS""",2026-09-04
"""APUS""",2026-09-24
"""APUS""",2026-10-07
…,…
"""XHLD""",2026-10-06
"""XPON""",2026-08-04
"""XPON""",2026-08-24


In [4]:
tickers_days = {
    row["ticker"]: row["days"]
    for row in (
        df_catalog
        .select(
            "ticker",
            pl.col("end").dt.strftime("%Y-%m-%d").alias("date"),
        )
        .group_by("ticker")
        .agg(
            pl.col("date")
            .unique()
            .sort(descending=True)
            .alias("days")
        )
        .sort("ticker")
        .to_dicts()
    )
}

tickers_days

{'AIXI': ['2026-08-28', '2026-08-07'],
 'APUS': ['2026-10-07', '2026-09-24', '2026-09-04'],
 'BAOS': ['2026-09-04', '2026-08-14'],
 'BEAT': ['2026-10-05', '2026-09-15'],
 'BIAF': ['2026-09-04', '2026-08-14'],
 'CAN': ['2026-08-21', '2026-07-31'],
 'CDTG': ['2026-09-04', '2026-08-14'],
 'CHAI': ['2026-08-28', '2026-08-07'],
 'CHPT': ['2026-09-03', '2026-08-14'],
 'CISS': ['2026-08-28', '2026-08-07'],
 'COOT': ['2026-08-31', '2026-08-11'],
 'CYAB': ['2026-08-28', '2026-08-07'],
 'FBGL': ['2026-09-22', '2026-09-02'],
 'FLYE': ['2026-10-08', '2026-09-18', '2026-09-01', '2026-08-12'],
 'GCDT': ['2026-09-23', '2026-09-03'],
 'GIPR': ['2026-08-25', '2026-08-24', '2026-08-04'],
 'GRI': ['2026-09-04', '2026-08-14'],
 'GRML': ['2026-09-25', '2026-09-04'],
 'IMCC': ['2026-09-22', '2026-09-02'],
 'IMRN': ['2026-09-04', '2026-08-14'],
 'LIDR': ['2026-09-01', '2026-08-12'],
 'MEDS': ['2026-10-08', '2026-09-18'],
 'NCPL': ['2026-09-21', '2026-09-01'],
 'NCRA': ['2026-08-31', '2026-08-11'],
 'NEOV': [

In [5]:

columns = ["datetime", "ticker", "open", "high", "low", "close", "volume"]
frames = []

for ticker, days in tickers_days.items():
    parquet_path = os.path.join(data_dir, f"{ticker}.parquet")

    if not os.path.isfile(parquet_path):
        raise FileNotFoundError(f"No existe el paquete Parquet para {ticker}: {parquet_path}")

    requested_days = [date.fromisoformat(day) for day in days]

    ticker_df = (
        pl.read_parquet(parquet_path)
        .filter(pl.col("datetime").dt.date().is_in(requested_days))
        .with_columns(pl.lit(ticker).alias("ticker"))
        .select(columns)
    )

    frames.append(ticker_df)

df = pl.concat(frames).sort(["ticker", "datetime"])
df

datetime,ticker,open,high,low,close,volume
"datetime[μs, UTC]",str,f64,f64,f64,f64,i64
2026-08-07 08:00:00 UTC,"""AIXI""",1.08,1.134,1.01,1.01,5617
2026-08-07 08:01:00 UTC,"""AIXI""",1.07,1.07,1.07,1.07,753
2026-08-07 08:02:00 UTC,"""AIXI""",1.05,1.05,1.05,1.05,142
2026-08-07 08:04:00 UTC,"""AIXI""",1.03,1.03,1.03,1.03,138
2026-08-07 08:06:00 UTC,"""AIXI""",1.0207,1.0207,1.0207,1.0207,974
…,…,…,…,…,…,…
2026-08-28 23:48:00 UTC,"""XTNT""",0.371,0.371,0.3666,0.3666,3638
2026-08-28 23:49:00 UTC,"""XTNT""",0.3709,0.371,0.3709,0.3709,5540
2026-08-28 23:57:00 UTC,"""XTNT""",0.3655,0.3699,0.3655,0.3699,703


In [6]:
GROUPS = ["ticker", "trading_day"]


def daily_vwap():
    return [
        (
            (pl.col("price") * pl.col("volume")).cum_sum().over(GROUPS)
            / pl.col("volume").cum_sum().over(GROUPS)
        ).alias("vwap_daily")
    ]


def opening_range_15m():
    opening_window = (
        (pl.col("datetime").dt.hour() == 9)
        & (pl.col("datetime").dt.minute() >= 30)
        & (pl.col("datetime").dt.minute() < 45)
    )
    range_ready = (
        (pl.col("datetime").dt.hour() > 9)
        | (
            (pl.col("datetime").dt.hour() == 9)
            & (pl.col("datetime").dt.minute() >= 45)
        )
    )

    range_high = pl.when(opening_window).then(pl.col("high")).max().over(GROUPS)
    range_low = pl.when(opening_window).then(pl.col("low")).min().over(GROUPS)

    return [
        pl.when(range_ready).then(range_high).otherwise(None).alias("orb_high_15m"),
        pl.when(range_ready).then(range_low).otherwise(None).alias("orb_low_15m"),
    ]


def median_vol(period=100):
    median_volume = (
        pl.col("volume").rolling_median(period, min_samples=period).over(GROUPS)
    )
    return [median_volume.alias(f"median_vol_{period}")]


def atr(period=14):
    previous_close = pl.col("close").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    return [
        true_range
        .rolling_mean(window_size=period, min_samples=period)
        .over(GROUPS)
        .alias(f"atr_{period}")
    ]


def bollinger_bands(period=20, std_dev=2.0):
    middle = pl.col("close").rolling_mean(period, min_samples=period).over(GROUPS)
    deviation = (
        pl.col("close")
        .rolling_std(period, min_samples=period, ddof=0)
        .over(GROUPS)
    )

    return [
        middle.alias(f"bb_middle_{period}"),
        (middle + deviation * std_dev).alias(f"bb_upper_{period}"),
        (middle - deviation * std_dev).alias(f"bb_lower_{period}"),
    ]


def rsi(period=14):
    change = pl.col("close").diff().over(GROUPS)
    gain = pl.when(change > 0).then(change).otherwise(0.0)
    loss = pl.when(change < 0).then(-change).otherwise(0.0)

    average_gain = (
        gain.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )
    average_loss = (
        loss.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )

    return [
        pl.when(average_loss == 0)
        .then(100.0)
        .when(average_gain == 0)
        .then(0.0)
        .otherwise(100.0 - (100.0 / (1.0 + average_gain / average_loss)))
        .alias(f"rsi_{period}")
    ]


def donchian_channels(period=20):
    upper = pl.col("high").rolling_max(period, min_samples=period).over(GROUPS)
    lower = pl.col("low").rolling_min(period, min_samples=period).over(GROUPS)

    return [
        upper.alias(f"donchian_upper_{period}"),
        lower.alias(f"donchian_lower_{period}"),
        ((upper + lower) / 2).alias(f"donchian_middle_{period}"),
    ]


def momentum(period=10):
    return [
        (
            pl.col("close")
            - pl.col("close").shift(period).over(GROUPS)
        ).alias(f"momentum_{period}")
    ]


def add_indicators(frame, indicator_functions):
    expressions = [
        expression
        for indicator_function in indicator_functions
        for expression in indicator_function()
    ]
    return frame.with_columns(expressions)


df_to_plot = (
    df
    .with_columns(
        pl.col("datetime").dt.convert_time_zone("America/New_York").alias("datetime")
    )
    .sort(["ticker", "datetime"])
    .with_columns(
        pl.col("datetime").dt.date().alias("trading_day"),
        ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("price"),
    )
)

df_to_plot = add_indicators(
    df_to_plot,
    [
        daily_vwap,
        opening_range_15m,
        atr,
        bollinger_bands,
        rsi,
        donchian_channels,
        momentum,
        median_vol,
    ],
)

df_to_plot

datetime,ticker,open,high,low,close,volume,trading_day,price,vwap_daily,orb_high_15m,orb_low_15m,atr_14,bb_middle_20,bb_upper_20,bb_lower_20,rsi_14,donchian_upper_20,donchian_lower_20,donchian_middle_20,momentum_10,median_vol_100
"datetime[μs, America/New_York]",str,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-08-07 04:00:00 EDT,"""AIXI""",1.08,1.134,1.01,1.01,5617,2026-08-07,1.051333,1.051333,null,null,null,null,null,null,null,null,null,null,null,null
2026-08-07 04:01:00 EDT,"""AIXI""",1.07,1.07,1.07,1.07,753,2026-08-07,1.07,1.05354,null,null,null,null,null,null,null,null,null,null,null,null
2026-08-07 04:02:00 EDT,"""AIXI""",1.05,1.05,1.05,1.05,142,2026-08-07,1.05,1.053463,null,null,null,null,null,null,null,null,null,null,null,null
2026-08-07 04:04:00 EDT,"""AIXI""",1.03,1.03,1.03,1.03,138,2026-08-07,1.03,1.052976,null,null,null,null,null,null,null,null,null,null,null,null
2026-08-07 04:06:00 EDT,"""AIXI""",1.0207,1.0207,1.0207,1.0207,974,2026-08-07,1.0207,1.048852,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-08-28 19:48:00 EDT,"""XTNT""",0.371,0.371,0.3666,0.3666,3638,2026-08-28,0.368067,0.402913,0.38,0.3397,0.001921,0.366845,0.372453,0.361237,42.937978,0.3717,0.364,0.36785,0.0019,2365.0
2026-08-28 19:49:00 EDT,"""XTNT""",0.3709,0.371,0.3709,0.3709,5540,2026-08-28,0.370933,0.40291,0.38,0.3397,0.002186,0.36699,0.372854,0.361126,51.620203,0.3717,0.364,0.36785,0.0062,2365.0
2026-08-28 19:57:00 EDT,"""XTNT""",0.3655,0.3699,0.3655,0.3699,703,2026-08-28,0.368433,0.40291,0.38,0.3397,0.002529,0.367185,0.373162,0.361208,49.72534,0.3717,0.364,0.36785,0.0052,2273.5


Conversion de data para backtesting

In [8]:
def get_data_to_backtesting(df,ticker,date):
    data = df_to_plot.filter(pl.col('ticker')==ticker).filter(pl.col('datetime').dt.date()==date)
    data.columns = [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]
    data = data.to_pandas().set_index('Datetime')
    return data    



In [9]:
from backtesting import Backtest, Strategy
import numpy as np


class OrbBreakout(Strategy):
    valor_R = 2

    def ORB_indicator(self):
        return np.vstack(
            (
                self.data.Orb_high_15m,
                self.data.Orb_low_15m,
            )
        )

    def vwap_indicator(self):
        return self.data.Vwap_daily

    def init(self):
        self.runner_activation_threshold = None
        self.runner_highest_high = None
        self.orb_high, self.orb_low = self.I(self.ORB_indicator)
        self.vwap = self.I(self.vwap_indicator)

    def next(self):
        runner_trade = next(
            (trade for trade in self.trades if trade.tag == "runner"),
            None,
        )
        if runner_trade is not None and self.runner_activation_threshold is not None:
            if self.runner_highest_high is None:
                if self.data.High[-1] >= self.runner_activation_threshold:
                    self.runner_highest_high = self.data.High[-1]
            else:
                self.runner_highest_high = max(
                    self.runner_highest_high,
                    self.data.High[-1],
                )

            if self.runner_highest_high is not None:
                trailing_stop = self.runner_highest_high - 2 * self.data.Atr_14[-1]
                if runner_trade.sl is None or trailing_stop > runner_trade.sl:
                    runner_trade.sl = trailing_stop

        orb_high = self.data.Orb_high_15m[-1]
        previous_orb_high = self.data.Orb_high_15m[-2]
        crossed_orb_high = (
            np.isfinite(orb_high)
            and np.isfinite(previous_orb_high)
            and self.data.Close[-1] > orb_high
            and self.data.Close[-2] <= previous_orb_high
        )

        recent_volumes = self.data.Volume[-4:-1]
        volume_surge = (
            len(recent_volumes) == 3
            and self.data.Volume[-1] > self.data.Median_vol_100[-1]
        )
        vwap_filter = self.data.Close < self.vwap * 1.05
        if crossed_orb_high and volume_surge and vwap_filter and not self.trades and not self.orders:
            atr = self.data.Atr_14[-1]
            entry_price = self.data.Close[-1]
            stop_loss = entry_price - atr * 2
            take_profit_1 = entry_price + atr * 2
            take_profit_2 = entry_price + atr * 4

            if np.isfinite(atr) and stop_loss > 0 and take_profit_1 > 0:
                size = int(self.valor_R / (entry_price - stop_loss))
                size_2 = int(size / 4)
                runner_size = int(size / 4)
                size_1 = size - size_2 - runner_size

                if size_1 > 0:
                    self.runner_activation_threshold = take_profit_2
                    self.runner_highest_high = None
                    self.buy(sl=stop_loss, tp=take_profit_1, size=size_1, tag="tp1")
                    self.buy(sl=stop_loss, tp=take_profit_2, size=size_2, tag="tp2")
                    self.buy(sl=stop_loss, size=runner_size, tag="runner")


d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [11]:
from pathlib import Path
from datetime import timedelta
import math


def strategy_name(strategy):
    """Return the class name for a Backtesting.py strategy class or instance."""
    return strategy.__name__ if isinstance(strategy, type) else type(strategy).__name__


def build_backtest_output_dir(config, strategy, ticker, trading_date):
    """Build the output directory without creating it."""
    day = trading_date.date() if isinstance(trading_date, datetime) else trading_date
    return (
        Path(config["data_dir"])
        / "output"
        / strategy_name(strategy)
        / str(ticker)
        / day.strftime("%Y%m%d")
    )


def backtesting_stats_to_scalar_metrics(stats):
    """Keep only scalar Backtesting.py statistics suitable for Polars/Parquet."""
    metrics = {}
    for key, value in stats.items():
        if str(key).startswith("_"):
            continue

        if isinstance(value, np.generic):
            value = value.item()

        if isinstance(value, (datetime, date)):
            value = value.isoformat()
        elif isinstance(value, timedelta):
            value = value.total_seconds()
        elif isinstance(value, float) and not math.isfinite(value):
            value = None
        elif not isinstance(value, (str, int, float, bool, type(None))):
            continue

        metrics[str(key)] = value
    return metrics

In [15]:
def run_and_persist_backtest(ticker, trading_date, strategy=OrbBreakout):
    """Run and persist one ticker/date backtest using the selected strategy settings."""
    day = trading_date.date() if isinstance(trading_date, datetime) else trading_date
    data = get_data_to_backtesting(df_to_plot, ticker, day)

    if data.empty:
        raise ValueError(f"No backtesting data available for {ticker} on {day:%Y-%m-%d}.")

    bt = Backtest(
        data,
        strategy,
        commission=(0.001, 0),
        cash=500,
        exclusive_orders=False,
    )
    stats = bt.run(valor_R=2)

    output_dir = build_backtest_output_dir(config, strategy, ticker, day)
    output_dir.mkdir(parents=True, exist_ok=True)

    result_row = {
        **backtesting_stats_to_scalar_metrics(stats),
        "strategy": strategy_name(strategy),
        "ticker": str(ticker),
        "date": day.isoformat(),
    }
    result = pl.DataFrame([result_row])
    result.write_parquet(output_dir / "stats.parquet")
    bt.plot(filename=str(output_dir / "backtest.html"), open_browser=False)
    return result


In [16]:
def run_and_persist_backtest_batch(df_runs, strategy=OrbBreakout):
    """Run, persist, and aggregate every ticker/date combination in df_runs."""
    results = []
    current_strategy = strategy_name(strategy)

    for run in df_runs.select("ticker", "date").iter_rows(named=True):
        ticker = str(run["ticker"])
        trading_date = run["date"]
        day = trading_date.date() if isinstance(trading_date, datetime) else trading_date

        try:
            result = run_and_persist_backtest(ticker, trading_date, strategy=strategy).with_columns(
                pl.lit("success").alias("status"),
                pl.lit(None, dtype=pl.Utf8).alias("error_message"),
            )
        except Exception as exc:
            result = pl.DataFrame(
                [
                    {
                        "strategy": current_strategy,
                        "ticker": ticker,
                        "date": day.isoformat(),
                        "status": "failed",
                        "error_message": str(exc),
                    }
                ]
            )

        results.append(result)

    if results:
        aggregated = pl.concat(results, how="diagonal_relaxed").sort(["ticker", "date"])
    else:
        aggregated = pl.DataFrame(
            schema={
                "strategy": pl.Utf8,
                "ticker": pl.Utf8,
                "date": pl.Utf8,
                "status": pl.Utf8,
                "error_message": pl.Utf8,
            }
        )

    output_dir = Path(config["data_dir"]) / "output" / current_strategy
    output_dir.mkdir(parents=True, exist_ok=True)
    aggregated.write_parquet(output_dir / "results.parquet")
    return aggregated


In [17]:
ticker = "VEEA"
trading_date = datetime.strptime("2026-10-09", "%Y-%m-%d")
backtest_result = run_and_persist_backtest(ticker, trading_date, strategy=OrbBreakout)
backtest_result


d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],Volatility (Ann.) [%],CAGR [%],Sharpe Ratio,Sortino Ratio,Calmar Ratio,Alpha [%],Beta,Max. Drawdown [%],Avg. Drawdown [%],Max. Drawdown Duration,Avg. Drawdown Duration,# Trades,Win Rate [%],Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,strategy,ticker,date
str,str,f64,f64,f64,f64,f64,f64,null,null,f64,null,null,null,f64,f64,f64,null,null,null,i64,null,null,null,null,null,null,null,null,null,null,str,str,str
"""2026-10-09T04:00:00-04:00""","""2026-10-09T19:59:00-04:00""",57540.0,0.0,500.0,500.0,0.0,-14.940577,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""VEEA""","""2026-10-09"""


In [18]:
df_backtest_results = run_and_persist_backtest_batch(df_backtest_runs, strategy=OrbBreakout)
df_backtest_results

d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))
C:\Users\FLODEV\AppData\Local\Temp\ipykernel_16096\183409061.py:16: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(valor_R=2)
d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))
d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))
d:\00_PROG\23_algo_trd\021_sm_framework\venv\Lib\site-packages\bokeh\util\serializ

Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Commissions [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],Volatility (Ann.) [%],CAGR [%],Sharpe Ratio,Sortino Ratio,Calmar Ratio,Alpha [%],Beta,Max. Drawdown [%],Avg. Drawdown [%],Max. Drawdown Duration,Avg. Drawdown Duration,# Trades,Win Rate [%],Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,strategy,ticker,date,status,error_message
str,str,f64,f64,f64,f64,f64,f64,f64,null,null,f64,null,null,null,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str
"""2026-08-07T04:00:00-04:00""","""2026-08-07T19:59:00-04:00""",57540.0,35.144928,494.560986,500.0,0.024,-1.087803,-4.545455,null,null,-99.751803,null,null,null,-1.047791,0.008803,-1.087803,-1.087803,36060.0,36060.0,12,8.333333,3.632298,-6.097585,-1.703347,11820.0,2100.0,0.153454,-1.669836,-2.625317,-0.542537,"""OrbBreakout""","""AIXI""","""2026-08-07""","""success""",null
"""2026-08-28T04:00:00-04:00""","""2026-08-28T19:59:00-04:00""",57540.0,0.0,499.1255,500.0,null,-0.1749,0.752587,null,null,-61.713484,null,null,null,-0.179022,0.005477,-0.24,-0.24,12360.0,12360.0,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""AIXI""","""2026-08-28""","""success""",null
"""2026-09-04T09:30:00-04:00""","""2026-09-04T16:32:00-04:00""",25320.0,0.0,500.0,500.0,null,0.0,-5.2,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""APUS""","""2026-09-04""","""success""",null
"""2026-09-24T04:00:00-04:00""","""2026-09-24T19:59:00-04:00""",57540.0,0.0,500.0,500.0,null,0.0,-21.456044,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""APUS""","""2026-09-24""","""success""",null
"""2026-10-07T04:00:00-04:00""","""2026-10-07T19:59:00-04:00""",57540.0,1.88172,498.4816,500.487,0.006,-0.30368,-19.143876,null,null,-81.138658,null,null,null,-0.291679,0.000627,-0.40069,-0.40069,30300.0,30300.0,3,0.0,-2.410436,-2.428253,-2.422314,780.0,780.0,0.0,-2.422314,-1.756943,null,"""OrbBreakout""","""APUS""","""2026-10-07""","""success""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026-10-06T04:00:00-04:00""","""2026-10-06T19:59:00-04:00""",57540.0,5.443038,500.697614,501.489986,0.012,0.139523,8.898712,null,null,114.829525,null,null,null,0.097847,0.004683,-0.722067,-0.336437,30060.0,11220.0,6,50.0,7.887735,-5.592719,0.028217,1380.0,1080.0,1.072092,0.201445,0.316662,0.127684,"""OrbBreakout""","""XHLD""","""2026-10-06""","""success""",null
"""2026-08-04T07:16:00-04:00""","""2026-08-04T16:05:00-04:00""",31740.0,0.0,500.0,500.0,null,0.0,7.368421,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""XPON""","""2026-08-04""","""success""",null
"""2026-08-24T05:17:00-04:00""","""2026-08-24T19:59:00-04:00""",52920.0,0.0,500.0,500.0,null,0.0,-29.998385,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""XPON""","""2026-08-24""","""success""",null


In [19]:
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

kpi_columns = [
    "Return [%]",
    "Buy & Hold Return [%]",
    "Win Rate [%]",
    "Avg. Trade [%]",
    "Avg. Drawdown [%]",
    "Expectancy [%]",
]
strategy_colors = {
    "OrbBreakout": "#1f77b4",
}

results_dir = Path(config["data_dir"]) / "output"
result_paths = {
    strategy: results_dir / strategy / "results.parquet"
    for strategy in strategy_colors
}
missing_files = [str(path) for path in result_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Cannot plot KPI comparisons because these result files are missing: "
        + ", ".join(missing_files)
    )

successful_results = {}
required_columns = ["status", *kpi_columns]
for strategy, path in result_paths.items():
    results = pl.read_parquet(path)
    missing_columns = [
        column for column in required_columns if column not in results.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Cannot plot {strategy} KPI comparisons because its results are missing: "
            + ", ".join(missing_columns)
        )

    successful_results[strategy] = results.filter(pl.col("status") == "success")
    if successful_results[strategy].height == 0:
        raise ValueError(
            f"Cannot plot {strategy} KPI comparisons because there are no successful results."
        )

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=kpi_columns,
)

for index, kpi in enumerate(kpi_columns):
    row = index // 3 + 1
    col = index % 3 + 1

    for strategy, color in strategy_colors.items():
        values = successful_results[strategy].get_column(kpi).drop_nulls().to_list()
        fig.add_trace(
            go.Histogram(
                x=values,
                name=strategy,
                legendgroup=strategy,
                showlegend=index == 0,
                marker_color=color,
                opacity=0.55,
            ),
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text=kpi, row=row, col=col)
    fig.update_yaxes(title_text="Count", row=row, col=col)

fig.update_layout(
    title="KPI distributions: OrbBreakout",
    barmode="overlay",
    height=700,
    width=1_300,
)
fig.show()

In [30]:
results

Start,End,Duration,Exposure Time [%],Equity Final [$],Equity Peak [$],Commissions [$],Return [%],Buy & Hold Return [%],Return (Ann.) [%],Volatility (Ann.) [%],CAGR [%],Sharpe Ratio,Sortino Ratio,Calmar Ratio,Alpha [%],Beta,Max. Drawdown [%],Avg. Drawdown [%],Max. Drawdown Duration,Avg. Drawdown Duration,# Trades,Win Rate [%],Best Trade [%],Worst Trade [%],Avg. Trade [%],Max. Trade Duration,Avg. Trade Duration,Profit Factor,Expectancy [%],SQN,Kelly Criterion,strategy,ticker,date,status,error_message
str,str,f64,f64,f64,f64,f64,f64,f64,null,null,f64,null,null,null,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str
"""2026-08-07T04:00:00-04:00""","""2026-08-07T19:59:00-04:00""",57540.0,35.144928,494.560986,500.0,0.024,-1.087803,-4.545455,null,null,-99.751803,null,null,null,-1.047791,0.008803,-1.087803,-1.087803,36060.0,36060.0,12,8.333333,3.632298,-6.097585,-1.703347,11820.0,2100.0,0.153454,-1.669836,-2.625317,-0.542537,"""OrbBreakout""","""AIXI""","""2026-08-07""","""success""",null
"""2026-08-28T04:00:00-04:00""","""2026-08-28T19:59:00-04:00""",57540.0,0.0,499.1255,500.0,null,-0.1749,0.752587,null,null,-61.713484,null,null,null,-0.179022,0.005477,-0.24,-0.24,12360.0,12360.0,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""AIXI""","""2026-08-28""","""success""",null
"""2026-09-04T09:30:00-04:00""","""2026-09-04T16:32:00-04:00""",25320.0,0.0,500.0,500.0,null,0.0,-5.2,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""APUS""","""2026-09-04""","""success""",null
"""2026-09-24T04:00:00-04:00""","""2026-09-24T19:59:00-04:00""",57540.0,0.0,500.0,500.0,null,0.0,-21.456044,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""APUS""","""2026-09-24""","""success""",null
"""2026-10-07T04:00:00-04:00""","""2026-10-07T19:59:00-04:00""",57540.0,1.88172,498.4816,500.487,0.006,-0.30368,-19.143876,null,null,-81.138658,null,null,null,-0.291679,0.000627,-0.40069,-0.40069,30300.0,30300.0,3,0.0,-2.410436,-2.428253,-2.422314,780.0,780.0,0.0,-2.422314,-1.756943,null,"""OrbBreakout""","""APUS""","""2026-10-07""","""success""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026-10-06T04:00:00-04:00""","""2026-10-06T19:59:00-04:00""",57540.0,5.443038,500.697614,501.489986,0.012,0.139523,8.898712,null,null,114.829525,null,null,null,0.097847,0.004683,-0.722067,-0.336437,30060.0,11220.0,6,50.0,7.887735,-5.592719,0.028217,1380.0,1080.0,1.072092,0.201445,0.316662,0.127684,"""OrbBreakout""","""XHLD""","""2026-10-06""","""success""",null
"""2026-08-04T07:16:00-04:00""","""2026-08-04T16:05:00-04:00""",31740.0,0.0,500.0,500.0,null,0.0,7.368421,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""XPON""","""2026-08-04""","""success""",null
"""2026-08-24T05:17:00-04:00""","""2026-08-24T19:59:00-04:00""",52920.0,0.0,500.0,500.0,null,0.0,-29.998385,null,null,0.0,null,null,null,0.0,0.0,-0.0,null,null,null,0,null,null,null,null,null,null,null,null,null,null,"""OrbBreakout""","""XPON""","""2026-08-24""","""success""",null
